# Notebook 2 · Análisis geográfico del consumo y la facturación — Electro Puno S.A.A.

**Objetivo.** Identificar qué provincias y distritos concentran consumo y facturación, si ambos rankings coinciden y qué diferencias hay entre *total* y *promedio por suministro*.

**Preguntas:** 1) ¿qué provincias concentran más consumo? 2) ¿qué distritos facturan más? 3) ¿coinciden consumo y facturación? 4) ¿qué zonas concentran la mayor participación? 5) ¿total vs. promedio por unidad? 6) ¿cómo se distribuyen los suministros?

**Filtros interactivos** (periodo, provincia, distrito, tarifa): están en un panel único; al moverlos se actualizan **todos** los gráficos de las secciones 3 a 8. Si tu VS Code no muestra widgets, pon `INTERACTIVO = False` en la sección 2 y se dibujará una vista fija.

**Requisitos:** `pip install duckdb pandas numpy plotly scipy ipywidgets`

> **Unidad de análisis.** Usamos **suministro‑mes** (un suministro observado en un mes). En 2023–2025 equivale a contar filas; en 2026 equivale a sumar la columna `clientes` (cada fila es un grupo). Con esa definición los **totales y conteos** de ambas etapas son compatibles. No calculamos medianas ni percentiles aquí porque en 2026 no hay suministros individuales.

In [1]:
# ============================================================
# CONFIGURACIÓN COMÚN (idéntica en los tres notebooks)
# Qué hace: importa librerías, localiza la raíz del proyecto,
# comprueba que existan los archivos y abre DuckDB.
# ============================================================
import sys, json, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

def verificar_paquetes(requeridos):
    """requeridos: {nombre_del_módulo: nombre_en_pip}"""
    import importlib
    faltan = []
    for modulo, paquete in requeridos.items():
        try:
            importlib.import_module(modulo)
        except ImportError:
            faltan.append(paquete)
    if faltan:
        raise ImportError(
            "Faltan paquetes en el entorno de este kernel: " + ", ".join(faltan)
            + "\nInstálalos con:\n    pip install " + " ".join(faltan)
            + "\ny reinicia el kernel."
        )

verificar_paquetes({'duckdb': 'duckdb', 'pandas': 'pandas', 'numpy': 'numpy', 'plotly': 'plotly', 'scipy': 'scipy', 'ipywidgets': 'ipywidgets'})

import duckdb
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
import ipywidgets as widgets
pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---------- 1) Localizar la raíz del proyecto (robusto en VS Code) ----------
def encontrar_raiz():
    """Busca hacia arriba (y un nivel hacia abajo) la carpeta que contiene
    preprocesamiento/silver. Sirve aunque el directorio de trabajo cambie."""
    inicios = [Path.cwd().resolve()]
    ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code define esta variable
    if ruta_nb:
        inicios.insert(0, Path(ruta_nb).resolve().parent)
    for inicio in inicios:
        for base in [inicio, *inicio.parents]:
            if (base / "preprocesamiento" / "silver").is_dir():
                return base
        for hijo in inicio.glob("*/preprocesamiento/silver"):
            return hijo.parent.parent
    raise FileNotFoundError(
        "No encuentro la carpeta 'preprocesamiento/silver'.\n"
        f"Directorio de trabajo actual: {Path.cwd()}\n"
        "Abre VS Code en la carpeta 'visualizacion-datos-trabajo' (o dentro de ella) "
        "y vuelve a ejecutar."
    )

RAIZ = encontrar_raiz()
SILVER = RAIZ / "preprocesamiento" / "silver"
RESULTADOS = RAIZ / "analisis" / "resultados"      # resultados intermedios (no son los datos originales)
RESULTADOS.mkdir(parents=True, exist_ok=True)

ARCHIVOS = {
    "clientes":   SILVER / "consumo_clientes.parquet",
    "agregado":   SILVER / "consumo_agregado.parquet",
    "resumen":    SILVER / "resumen_mensual.parquet",
    "rechazados": SILVER / "_rechazados.parquet",
}
OPCIONALES = {"rechazados"}
for nombre, ruta in ARCHIVOS.items():
    if not ruta.exists():
        if nombre in OPCIONALES:
            print(f"⚠️  Archivo opcional no encontrado: {ruta.name}")
        else:
            raise FileNotFoundError(f"Falta el archivo obligatorio: {ruta}")

# ---------- 2) Conexión DuckDB y vistas (no se carga nada en memoria) ----------
con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")   # menos memoria en agregaciones grandes

def _lit(ruta):                                       # escapa comillas simples para SQL
    return str(ruta.as_posix()).replace("'", "''")

for nombre, ruta in ARCHIVOS.items():
    if ruta.exists():
        con.execute(f"CREATE OR REPLACE VIEW {nombre} AS SELECT * FROM read_parquet('{_lit(ruta)}')")

def q(sql):
    """Ejecuta SQL en DuckDB y devuelve un DataFrame de pandas."""
    return con.execute(sql).df()

def sql_lista(valores):
    """Convierte una lista de textos en ('A','B') seguro para SQL."""
    if len(valores) == 0:
        return "(NULL)"
    return "(" + ", ".join("'" + str(v).replace("'", "''") + "'" for v in valores) + ")"

# ---------- 3) Validación de columnas ----------
COLUMNAS_OBLIGATORIAS = {
    "clientes": ["codigo_cliente","ubigeo","departamento","provincia","distrito","tarifa",
                 "periodo","fecha_alta","consumo_kwh","facturacion_soles"],
    "agregado": ["ubigeo","departamento","provincia","distrito","tarifa","periodo",
                 "consumo_kwh","facturacion_soles","clientes"],
    "resumen":  ["periodo","anio","mes","granularidad","n_suministros",
                 "consumo_kwh_total","facturacion_soles_total","consumo_kwh_promedio"],
}
for nombre, cols in COLUMNAS_OBLIGATORIAS.items():
    existentes = set(q(f"DESCRIBE SELECT * FROM {nombre}")["column_name"])
    faltan = [c for c in cols if c not in existentes]
    if faltan:
        raise KeyError(f"En '{ARCHIVOS[nombre].name}' faltan las columnas: {faltan}. "
                       f"Columnas encontradas: {sorted(existentes)}")

# ---------- 4) Huella de los datos (para reutilizar resultados guardados) ----------
HUELLA = hashlib.md5("|".join(
    f"{p.name}:{p.stat().st_size}:{p.stat().st_mtime_ns}" for p in ARCHIVOS.values() if p.exists()
).encode()).hexdigest()[:8]

def cachear(nombre, sql):
    """Si ya existe el resultado guardado con la misma huella de datos, lo lee;
    si no, ejecuta la consulta y lo guarda en analisis/resultados/ (Parquet)."""
    ruta = RESULTADOS / f"nb02_{nombre}_{HUELLA}.parquet"
    if ruta.exists():
        return con.execute(f"SELECT * FROM read_parquet('{_lit(ruta)}')").df()
    for viejo in RESULTADOS.glob(f"nb02_{nombre}_*.parquet"):
        viejo.unlink()                                 # elimina versiones obsoletas
    df = q(sql)
    con.register("_tmp_cache", df)
    con.execute(f"COPY _tmp_cache TO '{_lit(ruta)}' (FORMAT PARQUET)")
    con.unregister("_tmp_cache")
    return df

# ---------- 5) Constantes y utilidades compartidas ----------
SEMILLA = 42                                           # semilla para todo muestreo
MESES = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
COLOR_DETALLE  = "#1f77b4"    # azul: 2023-2025 (detalle por suministro)
COLOR_AGREGADO = "#ff7f0e"    # naranja: 2026 (agregado)
COLOR_CONSUMO  = "#2ca02c"    # verde: consumo (kWh)
COLOR_FACTURA  = "#d62728"    # rojo: facturación (S/)
# Fecha de alta <= este valor se trata como "dudosa" (valor por defecto/centinela probable).
FECHA_ALTA_DUDOSA_HASTA = "1989-01-01"

def fmt_n(x, d=0):
    return "s/d" if pd.isna(x) else f"{x:,.{d}f}"

def fmt_pct(x, d=1):
    return "s/d" if pd.isna(x) else f"{x:+.{d}f}%"

print(f"Raíz del proyecto : {RAIZ}")
print(f"Carpeta silver    : {SILVER}")
print(f"Resultados        : {RESULTADOS}")
print(f"Huella de datos   : {HUELLA}")
print("Archivos: " + ", ".join(f"{n} ({p.stat().st_size/1e6:,.1f} MB)" for n, p in ARCHIVOS.items() if p.exists()))

Raíz del proyecto : D:\PROYECTO\visualizacion-datos-trabajo
Carpeta silver    : D:\PROYECTO\visualizacion-datos-trabajo\preprocesamiento\silver
Resultados        : D:\PROYECTO\visualizacion-datos-trabajo\analisis\resultados
Huella de datos   : 222a6326
Archivos: clientes (100.1 MB), agregado (0.6 MB), resumen (0.0 MB), rechazados (0.0 MB)


## 1. Calidad del dato geográfico (¿qué clave territorial es confiable?)

Antes de comparar territorios comprobamos cómo se relacionan `ubigeo`, `provincia` y `distrito`. **No existen coordenadas** en los archivos; por eso no se inventa ningún mapa (ver sección 10).

La prueba clave: ¿cada nombre (provincia + distrito) corresponde a **un solo** `ubigeo`? Si no, hay filas cuyo código y cuyo nombre se contradicen y hay que decidir cuál usar como clave.

In [2]:
geo_filas = q("""
SELECT ubigeo, departamento, provincia, distrito, COUNT(*)::BIGINT AS suministros_mes FROM clientes GROUP BY 1, 2, 3, 4
UNION ALL
SELECT ubigeo, departamento, provincia, distrito, SUM(clientes)::BIGINT FROM agregado GROUP BY 1, 2, 3, 4
""")
geo_filas = geo_filas.groupby(["ubigeo", "departamento", "provincia", "distrito"], as_index=False)["suministros_mes"].sum()
tot_sm = geo_filas["suministros_mes"].sum()

print(f"Ubigeos distintos: {geo_filas['ubigeo'].nunique()} | pares (provincia, distrito) distintos: {geo_filas[['provincia','distrito']].drop_duplicates().shape[0]}")
hom = geo_filas[["provincia", "distrito"]].drop_duplicates().groupby("distrito").size()
print("Nombres de distrito que existen en más de una provincia (homónimos):", hom[hom > 1].index.tolist() or "ninguno")

por_nombre = geo_filas.groupby(["provincia", "distrito"])["ubigeo"].nunique()
print(f"Pares (provincia, distrito) con más de un ubigeo: {(por_nombre > 1).sum()} de {len(por_nombre)}")

modal = (geo_filas.sort_values("suministros_mes", ascending=False).drop_duplicates(["provincia", "distrito"])
         .set_index(["provincia", "distrito"])["ubigeo"])
geo_filas["ubigeo_modal"] = [modal[(p, d)] for p, d in zip(geo_filas["provincia"], geo_filas["distrito"])]
disc = geo_filas[geo_filas["ubigeo"] != geo_filas["ubigeo_modal"]]
pct_disc = 100 * disc["suministros_mes"].sum() / tot_sm
print(f"Suministros-mes cuyo ubigeo NO coincide con el ubigeo habitual de su nombre de distrito: {disc['suministros_mes'].sum():,} ({pct_disc:.2f}%)")
display(disc.sort_values("suministros_mes", ascending=False).head(6)[["provincia", "distrito", "ubigeo", "ubigeo_modal", "suministros_mes"]])

print("DECISIÓN: se usa el PAR (provincia, distrito) como clave territorial; el ubigeo habitual de cada par se conserva solo como referencia.")
print("La sensibilidad de esta decisión se comprueba en la sección 11 (rankings por nombre vs. por ubigeo).")

fuera = geo_filas[geo_filas["departamento"] != "PUNO"]
print("Registros fuera del departamento de Puno (se excluyen de rankings):")
display(fuera)

geo = geo_filas[geo_filas["departamento"] == "PUNO"][["provincia", "distrito"]].drop_duplicates().copy()
geo["zona"] = geo["provincia"] + " | " + geo["distrito"]
geo["etiqueta"] = geo["distrito"] + " (" + geo["provincia"].str.title() + ")"
geo["ubigeo_modal"] = [modal[(p, d)] for p, d in zip(geo["provincia"], geo["distrito"])]
GEO = geo.set_index("zona")
ETIQUETA = GEO["etiqueta"].to_dict()
PROVINCIAS = sorted(GEO["provincia"].unique())
print(f"Se usarán {len(GEO)} distritos de {len(PROVINCIAS)} provincias de Puno.")

Ubigeos distintos: 112 | pares (provincia, distrito) distintos: 110
Nombres de distrito que existen en más de una provincia (homónimos): ['SANTA ROSA']
Pares (provincia, distrito) con más de un ubigeo: 99 de 110
Suministros-mes cuyo ubigeo NO coincide con el ubigeo habitual de su nombre de distrito: 131,476 (0.92%)


,provincia,distrito,ubigeo,ubigeo_modal,suministros_mes
86,PUNO,PUNO,210111,210101,27683
344,SAN ROMAN,SAN MIGUEL,211101,211105,24386
106,AZANGARO,ACHAYA,210202,210703,13551
83,PUNO,PUNO,210110,210101,7530
74,PUNO,COATA,210108,210107,5092
306,HUANCANE,INCHUPALLA,211003,210604,4843


DECISIÓN: se usa el PAR (provincia, distrito) como clave territorial; el ubigeo habitual de cada par se conserva solo como referencia.
La sensibilidad de esta decisión se comprueba en la sección 11 (rankings por nombre vs. por ubigeo).
Registros fuera del departamento de Puno (se excluyen de rankings):


,ubigeo,departamento,provincia,distrito,suministros_mes,ubigeo_modal
1,150140,LIMA,LIMA,SANTIAGO DE SURCO,13,150140


Se usarán 109 distritos de 13 provincias de Puno.


## 2. Cubo de datos y panel de filtros

Resumimos **una sola vez** los 11.4 millones de filas + los registros agregados en un cubo pequeño `periodo × provincia × distrito × tarifa` (con DuckDB; se guarda en `analisis/resultados/` para no recalcular). Todos los filtros trabajan sobre ese cubo en pandas, por lo que responden al instante.

Columnas del cubo: `suministros_mes`, `consumo_kwh`, `facturacion_soles`, `suministros_kwh_cero`.

**Tarifas sin kWh informado.** Si una tarifa tiene `consumo_kwh = 0` en prácticamente todas sus filas pero factura, su consumo **no está medido en estos archivos**. Se detectan automáticamente; para promedios de kWh se excluyen del denominador.

In [3]:
CUBO = cachear("cubo_geo", """
SELECT *, 'cliente' AS granularidad FROM (
  SELECT periodo, departamento, provincia, distrito, tarifa,
         COUNT(*)::BIGINT AS suministros_mes,
         SUM(consumo_kwh) AS consumo_kwh,
         SUM(facturacion_soles) AS facturacion_soles,
         SUM(CASE WHEN consumo_kwh = 0 THEN 1 ELSE 0 END)::BIGINT AS suministros_kwh_cero
  FROM clientes GROUP BY 1, 2, 3, 4, 5)
UNION ALL
SELECT *, 'agregado' AS granularidad FROM (
  SELECT periodo, departamento, provincia, distrito, tarifa,
         SUM(clientes)::BIGINT AS suministros_mes,
         SUM(consumo_kwh) AS consumo_kwh,
         SUM(facturacion_soles) AS facturacion_soles,
         SUM(CASE WHEN consumo_kwh = 0 THEN clientes ELSE 0 END)::BIGINT AS suministros_kwh_cero
  FROM agregado GROUP BY 1, 2, 3, 4, 5)
""")
print(f"Cubo: {len(CUBO):,} filas (de ~11.5 millones de registros originales).")

# --- Control de integridad contra el resumen mensual ---
r = q("SELECT periodo, n_suministros, consumo_kwh_total, facturacion_soles_total FROM resumen").set_index("periodo")
c = CUBO.groupby("periodo")[["suministros_mes", "consumo_kwh", "facturacion_soles"]].sum()
assert np.allclose(c["suministros_mes"], r["n_suministros"]) and np.allclose(c["consumo_kwh"], r["consumo_kwh_total"], rtol=1e-6) \
   and np.allclose(c["facturacion_soles"], r["facturacion_soles_total"], rtol=1e-6), "El cubo no reproduce el resumen mensual"
print("✅ El cubo reproduce el resumen mensual (suministros, kWh, soles).")

# --- Tarifas sin kWh informado ---
et = CUBO.groupby("tarifa").agg(s=("suministros_mes", "sum"), z=("suministros_kwh_cero", "sum"), f=("facturacion_soles", "sum"))
et["pct_cero"] = 100 * et["z"] / et["s"]
TARIFAS_SIN_KWH = et.index[et["pct_cero"] >= 99.0].tolist()
TARIFAS = et.sort_values("f", ascending=False).index.tolist()
print("Tarifas sin kWh informado (≥99 % de filas con kWh = 0):", TARIFAS_SIN_KWH)

CUBO["suministros_mes_kwh"] = np.where(CUBO["tarifa"].isin(TARIFAS_SIN_KWH), 0, CUBO["suministros_mes"])
CUBO["fact_con_kwh"] = np.where(CUBO["tarifa"].isin(TARIFAS_SIN_KWH), 0.0, CUBO["facturacion_soles"])

# --- Separar lo que está fuera de Puno (no entra en rankings territoriales) ---
CUBO["zona"] = CUBO["provincia"] + " | " + CUBO["distrito"]
fuera_c = CUBO[~CUBO["zona"].isin(GEO.index)]
print(f"Filas del cubo fuera de Puno (excluidas de rankings): {len(fuera_c)} | suministros-mes: {fuera_c['suministros_mes'].sum():,} | "
      f"facturación: S/ {fuera_c['facturacion_soles'].sum():,.0f}")
CUBO = CUBO[CUBO["zona"].isin(GEO.index)].copy()
CUBO["etiqueta"] = CUBO["zona"].map(ETIQUETA)
PERIODOS = sorted(CUBO["periodo"].unique())
print(f"Periodos con datos: {len(PERIODOS)} ({PERIODOS[0]} a {PERIODOS[-1]})")

Cubo: 27,216 filas (de ~11.5 millones de registros originales).
✅ El cubo reproduce el resumen mensual (suministros, kWh, soles).
Tarifas sin kWh informado (≥99 % de filas con kWh = 0): ['BT2', 'BT3', 'BT5A', 'MT2', 'MT3']
Filas del cubo fuera de Puno (excluidas de rankings): 13 | suministros-mes: 13 | facturación: S/ 729
Periodos con datos: 41 (202301 a 202608)


In [4]:
# ---------------- Panel de filtros ----------------
INTERACTIVO = True        # ← pon False si tu VS Code no muestra widgets: se dibujará una vista fija

opc_periodo = [(f"{str(p)[:4]}-{str(p)[4:]}", p) for p in PERIODOS]
w_periodo = widgets.SelectionRangeSlider(options=opc_periodo, index=(0, len(opc_periodo) - 1), description="Periodo",
                                         layout=widgets.Layout(width="700px"), continuous_update=False)
w_prov  = widgets.Dropdown(options=["Todas"] + PROVINCIAS, value="Todas", description="Provincia")
w_dist  = widgets.Dropdown(options=[("Todos", None)], value=None, description="Distrito")
w_tarifa = widgets.SelectMultiple(options=TARIFAS, value=tuple(TARIFAS), rows=min(len(TARIFAS), 8), description="Tarifa")
w_topn  = widgets.IntSlider(value=12, min=5, max=40, step=1, description="Top N")
w_min   = widgets.IntSlider(value=1000, min=0, max=20000, step=500, description="Mín. suministros-mes", style={"description_width": "initial"})
w_metrica = widgets.RadioButtons(options=[("Consumo (kWh)", "consumo_kwh"), ("Facturación (S/)", "facturacion_soles"),
                                          ("Suministros-mes", "suministros_mes")], value="consumo_kwh", description="Métrica (sec. 6)")

def _actualizar_distritos(cambio):
    if w_prov.value == "Todas":
        sub = GEO
    else:
        sub = GEO[GEO["provincia"] == w_prov.value]
    w_dist.options = [("Todos", None)] + [(r["etiqueta"], z) for z, r in sub.sort_values("etiqueta").iterrows()]
    w_dist.value = None
w_prov.observe(_actualizar_distritos, names="value")
_actualizar_distritos(None)

CONTROLES = {"rango": w_periodo, "prov": w_prov, "dist": w_dist, "tarifas": w_tarifa, "topn": w_topn, "minsum": w_min, "metrica": w_metrica}
panel = widgets.VBox([w_periodo, widgets.HBox([w_prov, w_dist]), widgets.HBox([w_tarifa, widgets.VBox([w_topn, w_min, w_metrica])])])
if INTERACTIVO:
    display(panel)

# ---------------- Funciones de apoyo ----------------
COLOR_PROV = {p: px.colors.qualitative.Alphabet[i % 26] for i, p in enumerate(PROVINCIAS)}

def leer(kw):
    p0, p1 = kw["rango"]
    return dict(p0=p0, p1=p1, tarifas=list(kw["tarifas"]) or TARIFAS, prov=kw["prov"], zona=kw["dist"],
                topn=int(kw["topn"]), minsum=int(kw["minsum"]), metrica=kw["metrica"])

def preparar(F, aplicar_provincia=False):
    d = CUBO[(CUBO["periodo"] >= F["p0"]) & (CUBO["periodo"] <= F["p1"]) & (CUBO["tarifa"].isin(F["tarifas"]))]
    if aplicar_provincia and F["prov"] != "Todas":
        d = d[d["provincia"] == F["prov"]]
    return d

def agregar(d, claves):
    g = (d.groupby(claves, as_index=False)
           .agg(suministros_mes=("suministros_mes", "sum"), consumo_kwh=("consumo_kwh", "sum"),
                facturacion_soles=("facturacion_soles", "sum"), suministros_mes_kwh=("suministros_mes_kwh", "sum"),
                fact_con_kwh=("fact_con_kwh", "sum"), periodos=("periodo", "nunique")))
    g["kwh_por_suministro"] = np.where(g["suministros_mes_kwh"] > 0, g["consumo_kwh"] / g["suministros_mes_kwh"].replace(0, np.nan), np.nan)
    g["soles_por_suministro"] = g["facturacion_soles"] / g["suministros_mes"].replace(0, np.nan)
    g["soles_por_kwh"] = np.where(g["consumo_kwh"] > 0, g["fact_con_kwh"] / g["consumo_kwh"].replace(0, np.nan), np.nan)
    return g

def encabezado(F, d):
    gran = sorted(d["granularidad"].unique())
    txt = {"cliente": "detalle por suministro (2023–2025)", "agregado": "agregado por grupos (2026)"}
    nper = d["periodo"].nunique()
    print(f"► Periodos: {F['p0']} a {F['p1']} ({nper} meses con datos) | Granularidad: " + " + ".join(txt[g] for g in gran))
    if len(gran) > 1:
        print("  ⚠ Se combinan detalle y agregado: válido para TOTALES y conteos de suministros-mes; no se calculan medianas.")
    print(f"► Tarifas: {len(F['tarifas'])} de {len(TARIFAS)} | Provincia: {F['prov']} | Distrito: {ETIQUETA.get(F['zona'], 'Todos')}")
    incl = [t for t in F["tarifas"] if t in TARIFAS_SIN_KWH]
    if incl:
        print(f"  ⚠ Incluye tarifas sin kWh informado {incl}: su facturación suma, pero su consumo NO está medido (el consumo total queda subestimado).")

def mostrar(vista):
    """Con widgets: enlaza la vista a los controles (se redibuja al cambiar un filtro). Sin widgets: dibuja una vez."""
    if INTERACTIVO:
        display(widgets.interactive_output(vista, CONTROLES))
    else:
        vista(**{k: (v.value) for k, v in CONTROLES.items()})

def gini(x):
    x = np.sort(np.asarray(x, dtype=float)); x = x[x >= 0]
    if len(x) == 0 or x.sum() == 0: return np.nan
    cum = np.cumsum(x)
    return (len(x) + 1 - 2 * cum.sum() / cum[-1]) / len(x)

## 3. Provincias: consumo y facturación (preguntas 1 y 4)

**Indicadores:** total de kWh y de S/ por provincia y su **participación %** sobre el total filtrado.
**Gráficos:** dos barras horizontales (consumo y facturación). La provincia elegida en el filtro se resalta en naranja.
**Cómo leerlo:** pase el cursor para ver el valor y la participación. Un orden distinto entre ambas barras sugiere diferencias de precio, tarifa o cargos entre provincias.
**Limitación:** total ≠ promedio; una provincia grande por número de clientes aparece arriba aunque cada cliente consuma poco.

In [5]:
def vista_provincias(**kw):
    F = leer(kw); d = preparar(F); encabezado(F, d)
    g = agregar(d, ["provincia"])
    g["part_consumo"] = 100 * g["consumo_kwh"] / g["consumo_kwh"].sum()
    g["part_fact"] = 100 * g["facturacion_soles"] / g["facturacion_soles"].sum()
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Consumo total (kWh)", "Facturación total (S/)"), horizontal_spacing=0.22)
    for j, (col, part, color) in enumerate([("consumo_kwh", "part_consumo", COLOR_CONSUMO), ("facturacion_soles", "part_fact", COLOR_FACTURA)], start=1):
        s = g.sort_values(col)
        colores = np.where(s["provincia"] == F["prov"], "#ff7f0e", color)
        fig.add_trace(go.Bar(y=s["provincia"], x=s[col], orientation="h", marker_color=colores, showlegend=False,
                             customdata=np.stack([s[part], s["suministros_mes"]], axis=1),
                             hovertemplate="<b>%{y}</b><br>%{x:,.0f}<br>Participación: %{customdata[0]:.1f}%<br>Suministros-mes: %{customdata[1]:,.0f}<extra></extra>"), row=1, col=j)
    fig.update_xaxes(title_text="kWh", row=1, col=1, tickformat=",.0f"); fig.update_xaxes(title_text="Soles (S/)", row=1, col=2, tickformat=",.0f")
    fig.update_layout(title="Consumo y facturación por provincia (según filtros)", height=520)
    fig.show()
mostrar(vista_provincias)

Output()

## 4. Distritos: rankings de consumo y facturación (pregunta 2)

**Indicador:** Top N de distritos por kWh y por S/, dentro de la provincia seleccionada (o todas).
**Cómo leerlo:** el color indica la provincia. Si un distrito aparece en un ranking y no en el otro, mire la sección 5.

In [6]:
def vista_distritos(**kw):
    F = leer(kw); d = preparar(F, aplicar_provincia=True); encabezado(F, d)
    g = agregar(d, ["zona", "etiqueta", "provincia"])
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Top {F['topn']} por consumo (kWh)", f"Top {F['topn']} por facturación (S/)"), horizontal_spacing=0.32)
    for j, col in enumerate(["consumo_kwh", "facturacion_soles"], start=1):
        s = g.nlargest(F["topn"], col).sort_values(col)
        fig.add_trace(go.Bar(y=s["etiqueta"], x=s[col], orientation="h", marker_color=[COLOR_PROV[p] for p in s["provincia"]], showlegend=False,
                             customdata=np.stack([100 * s[col] / g[col].sum(), s["suministros_mes"]], axis=1),
                             hovertemplate="<b>%{y}</b><br>%{x:,.0f}<br>Participación (del filtro): %{customdata[0]:.2f}%<br>Suministros-mes: %{customdata[1]:,.0f}<extra></extra>"), row=1, col=j)
    fig.update_xaxes(title_text="kWh", row=1, col=1, tickformat=",.0f"); fig.update_xaxes(title_text="Soles (S/)", row=1, col=2, tickformat=",.0f")
    fig.update_layout(title="Distritos con mayor consumo y facturación (color = provincia)", height=max(480, 26 * F["topn"] + 200))
    fig.show()
mostrar(vista_distritos)

Output()

## 5. ¿Coinciden consumo y facturación? (pregunta 3)

**Indicadores:** correlación de rangos de Spearman entre distritos, número de distritos que están en **ambos** Top N, y distritos con mayor diferencia de posición.
**Gráfico:** dispersión consumo vs. facturación por distrito (ejes logarítmicos, tamaño = suministros‑mes).
**Cómo leerlo:** puntos por **encima** de la nube (más soles para el mismo kWh) pueden reflejar mezcla tarifaria, cargos fijos o tarifas sin kWh informado (p. ej. media tensión); puntos por **debajo**, lo contrario.
**Limitación:** la facturación incluye tarifas cuyo kWh no se informa; por eso se muestra también S/kWh solo con tarifas que sí informan.

In [7]:
def vista_coincidencia(**kw):
    F = leer(kw); d = preparar(F, aplicar_provincia=True); encabezado(F, d)
    g = agregar(d, ["zona", "etiqueta", "provincia"])
    g = g[(g["consumo_kwh"] > 0) & (g["facturacion_soles"] > 0)].copy()
    if len(g) < 3:
        print("Pocos distritos con datos para esta combinación de filtros."); return
    rho, _ = stats.spearmanr(g["consumo_kwh"], g["facturacion_soles"])
    n = F["topn"]
    top_c = set(g.nlargest(n, "consumo_kwh")["zona"]); top_f = set(g.nlargest(n, "facturacion_soles")["zona"])
    print(f"Spearman (consumo vs. facturación, {len(g)} distritos) = {rho:.3f} | Distritos presentes en AMBOS Top {n}: {len(top_c & top_f)} de {n}")
    g["rank_consumo"] = g["consumo_kwh"].rank(ascending=False, method="min").astype(int)
    g["rank_fact"] = g["facturacion_soles"].rank(ascending=False, method="min").astype(int)
    g["dif_posicion"] = g["rank_fact"] - g["rank_consumo"]

    fig = px.scatter(g, x="consumo_kwh", y="facturacion_soles", size="suministros_mes", color="provincia", color_discrete_map=COLOR_PROV,
                     log_x=True, log_y=True, size_max=45, hover_name="etiqueta",
                     hover_data={"consumo_kwh": ":,.0f", "facturacion_soles": ":,.0f", "suministros_mes": ":,.0f", "soles_por_kwh": ":.3f",
                                 "rank_consumo": True, "rank_fact": True, "provincia": False},
                     labels={"consumo_kwh": "Consumo total (kWh, escala log)", "facturacion_soles": "Facturación total (S/, escala log)",
                             "suministros_mes": "Suministros-mes", "soles_por_kwh": "S/ por kWh (solo tarifas con kWh)",
                             "rank_consumo": "Posición por consumo", "rank_fact": "Posición por facturación", "provincia": "Provincia"})
    fig.update_layout(title="Consumo vs. facturación por distrito (tamaño = suministros-mes)", height=560)
    fig.show()
    t = g.reindex(g["dif_posicion"].abs().sort_values(ascending=False).index).head(8)
    print("Distritos con mayor diferencia entre su posición por facturación y por consumo (positivo = más alto en consumo que en facturación):")
    display(t[["etiqueta", "rank_consumo", "rank_fact", "dif_posicion", "soles_por_kwh"]].rename(columns={"soles_por_kwh": "S/ por kWh"}).style.format({"S/ por kWh": "{:.3f}"}))
mostrar(vista_coincidencia)

Output()

## 6. Participación porcentual (pregunta 4)

**Indicador:** participación (%) de cada zona en el total de la métrica elegida (botones *Métrica*).
**Gráficos:** (a) **Pareto** de distritos: barras = participación, línea = acumulada; (b) **treemap** provincia → distrito: el área es la métrica.
**Cómo leerlo:** si pocas zonas suman gran parte del total, la actividad está concentrada geográficamente.

In [8]:
def vista_participacion(**kw):
    F = leer(kw); d = preparar(F, aplicar_provincia=True); encabezado(F, d)
    col = F["metrica"]; nombre = {"consumo_kwh": "Consumo (kWh)", "facturacion_soles": "Facturación (S/)", "suministros_mes": "Suministros-mes"}[col]
    g = agregar(d, ["zona", "etiqueta", "distrito", "provincia"]); g = g[g[col] > 0].sort_values(col, ascending=False)
    if g.empty:
        print("Sin datos para esta combinación de filtros."); return
    g["part"] = 100 * g[col] / g[col].sum(); g["acum"] = g["part"].cumsum()
    s = g.head(max(F["topn"], 10))
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Bar(x=s["etiqueta"], y=s["part"], name="Participación (%)", marker_color=[COLOR_PROV[p] for p in s["provincia"]],
                         hovertemplate="<b>%{x}</b><br>%{y:.2f}% del total<extra></extra>"), secondary_y=False)
    fig.add_trace(go.Scatter(x=s["etiqueta"], y=s["acum"], mode="lines+markers", name="Acumulado (%)", line=dict(color="black"),
                             hovertemplate="%{x}<br>Acumulado: %{y:.1f}%<extra></extra>"), secondary_y=True)
    fig.update_yaxes(title_text="Participación (%)", secondary_y=False); fig.update_yaxes(title_text="Acumulado (%)", range=[0, 100], secondary_y=True)
    fig.update_xaxes(tickangle=-45)
    fig.update_layout(title=f"Pareto de distritos — {nombre}", height=520)
    fig.show()
    for k in (5, 10, 20):
        print(f"Los {k} distritos principales concentran {g['part'].head(k).sum():.1f}% de: {nombre}")
    tm = g.assign(valor=g[col])
    fig2 = px.treemap(tm, path=[px.Constant("Puno"), "provincia", "distrito"], values="valor", color="provincia", color_discrete_map={**COLOR_PROV, "(?)": "lightgray"},
                      title=f"Treemap provincia → distrito ({nombre})")
    fig2.update_traces(hovertemplate="<b>%{label}</b><br>%{value:,.0f}<br>%{percentRoot:.1%} del total<extra></extra>")
    fig2.update_layout(height=560)
    fig2.show()
mostrar(vista_participacion)

Output()

## 7. Total vs. promedio por suministro‑mes (pregunta 5)

**Indicador:** `kWh por suministro‑mes = kWh totales / suministros‑mes` **solo con tarifas que informan kWh** (para no diluir el promedio con ceros que significan "no medido").
**Gráfico:** dispersión total (eje X, log) vs. promedio por suministro (eje Y).
**Cómo leerlo:** zonas a la **derecha y abajo** son grandes en volumen pero con consumo individual bajo; zonas **arriba** tienen clientes de mayor consumo (p. ej. usuarios comerciales o industriales). Un distrito grande no implica consumo alto por cliente.
**Filtro clave:** *Mín. suministros‑mes* evita promedios de zonas con muy pocas observaciones.

In [10]:
def vista_total_vs_promedio(**kw):
    F = leer(kw); d = preparar(F, aplicar_provincia=True); encabezado(F, d)
    g = agregar(d, ["zona", "etiqueta", "provincia"])
    g = g[(g["suministros_mes_kwh"] >= max(F["minsum"], 1)) & g["kwh_por_suministro"].notna() & (g["consumo_kwh"] > 0)].copy()
    if len(g) < 3:
        print("Pocos distritos superan el mínimo de suministros-mes: baje el filtro."); return
    mediana = g["kwh_por_suministro"].median()
    fig = px.scatter(g, x="consumo_kwh", y="kwh_por_suministro", size="suministros_mes", color="provincia", color_discrete_map=COLOR_PROV,
                     log_x=True, size_max=40, hover_name="etiqueta",
                     hover_data={"consumo_kwh": ":,.0f", "kwh_por_suministro": ":.1f", "soles_por_suministro": ":.1f", "suministros_mes": ":,.0f", "provincia": False},
                     labels={"consumo_kwh": "Consumo total (kWh, escala log)", "kwh_por_suministro": "kWh por suministro-mes",
                             "soles_por_suministro": "S/ por suministro-mes", "suministros_mes": "Suministros-mes", "provincia": "Provincia"})
    fig.add_hline(y=mediana, line_dash="dot", line_color="gray", annotation_text=f"mediana de distritos: {mediana:.1f} kWh", annotation_position="top left")
    fig.update_layout(title="Consumo total vs. consumo promedio por suministro-mes (distritos)", height=560)
    fig.show()
    g["rank_total"] = g["consumo_kwh"].rank(ascending=False, method="min").astype(int)
    g["rank_promedio"] = g["kwh_por_suministro"].rank(ascending=False, method="min").astype(int)
    rho, _ = stats.spearmanr(g["consumo_kwh"], g["kwh_por_suministro"])
    print(f"Spearman entre consumo TOTAL y consumo PROMEDIO por suministro-mes = {rho:.3f} (cerca de 0 → el total no anticipa el promedio).")
    t = g.nlargest(F["topn"], "consumo_kwh")[["etiqueta", "consumo_kwh", "suministros_mes", "kwh_por_suministro", "rank_total", "rank_promedio"]]
    print(f"Top {F['topn']} por consumo total y su posición según consumo promedio:")
    display(t.style.format({"consumo_kwh": "{:,.0f}", "suministros_mes": "{:,.0f}", "kwh_por_suministro": "{:.1f}"}))
mostrar(vista_total_vs_promedio)

Output()

## 8. Distribución de los suministros y concentración (pregunta 6)

**Indicadores:** suministros promedio por mes en cada zona; **coeficiente de Gini** (0 = reparto parejo, 1 = todo en una zona), **índice HHI** (suma de participaciones² × 10 000) y participación de los 5 mayores distritos.
**Gráficos:** barras por provincia y **curvas de Lorenz** por distrito para consumo, facturación y suministros.
**Cómo leerlo:** cuanto más se aleja la curva de la diagonal, más concentrada está la variable.
**Limitación:** "suministros" = suministros‑mes (se promedia entre los meses con datos del filtro).

In [11]:
def vista_distribucion(**kw):
    F = leer(kw); d = preparar(F, aplicar_provincia=False); encabezado(F, d)
    nper = max(d["periodo"].nunique(), 1)
    gp = agregar(d, ["provincia"]); gp["susc_prom_mes"] = gp["suministros_mes"] / nper
    gp["part_susc"] = 100 * gp["suministros_mes"] / gp["suministros_mes"].sum()
    s = gp.sort_values("susc_prom_mes")
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Suministros promedio por mes y provincia", "Curvas de Lorenz entre distritos"), column_widths=[0.5, 0.5], horizontal_spacing=0.12)
    fig.add_trace(go.Bar(y=s["provincia"], x=s["susc_prom_mes"], orientation="h", marker_color=[COLOR_PROV[p] for p in s["provincia"]], showlegend=False,
                         customdata=s["part_susc"], hovertemplate="<b>%{y}</b><br>%{x:,.0f} suministros/mes<br>%{customdata:.1f}% del total<extra></extra>"), row=1, col=1)
    gd = agregar(d, ["zona"])
    filas = []
    for col, nombre, color in [("consumo_kwh", "Consumo", COLOR_CONSUMO), ("facturacion_soles", "Facturación", COLOR_FACTURA), ("suministros_mes", "Suministros", COLOR_DETALLE)]:
        x = np.sort(gd[col].clip(lower=0).values)
        if x.sum() == 0: continue
        cum = np.insert(np.cumsum(x) / x.sum(), 0, 0); pos = np.linspace(0, 1, len(cum))
        fig.add_trace(go.Scatter(x=pos, y=cum, mode="lines", name=f"{nombre} (Gini {gini(x):.2f})", line=dict(color=color),
                                 hovertemplate=f"{nombre}<br>%{{x:.0%}} de distritos (los menores) acumulan %{{y:.1%}}<extra></extra>"), row=1, col=2)
        sh = x / x.sum()
        filas.append({"variable": nombre, "Gini": gini(x), "HHI": 10000 * (sh ** 2).sum(), "% de los 5 mayores distritos": 100 * np.sort(sh)[::-1][:5].sum(), "distritos": len(x)})
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Igualdad perfecta", line=dict(color="gray", dash="dot")), row=1, col=2)
    fig.update_xaxes(title_text="Suministros por mes", tickformat=",.0f", row=1, col=1)
    fig.update_xaxes(title_text="Proporción acumulada de distritos", row=1, col=2); fig.update_yaxes(title_text="Proporción acumulada de la variable", row=1, col=2)
    fig.update_layout(height=520, legend=dict(orientation="h", y=-0.2))
    fig.show()
    display(pd.DataFrame(filas).style.format({"Gini": "{:.3f}", "HHI": "{:,.0f}", "% de los 5 mayores distritos": "{:.1f}", "distritos": "{:,.0f}"}))
mostrar(vista_distribucion)

Output()

## 9. Ficha territorial (usa el filtro **Distrito**)

Si elige un distrito, aquí ve su evolución mensual y su mezcla tarifaria; si solo elige provincia, ve la provincia; si no elige nada, todo Puno. Así el filtro de distrito **actúa realmente** sobre un gráfico. Los meses sin datos aparecen como huecos.

In [12]:
def vista_ficha(**kw):
    F = leer(kw); d = preparar(F)
    if F["zona"]:
        d = d[d["zona"] == F["zona"]]; titulo = ETIQUETA.get(F["zona"], F["zona"])
    elif F["prov"] != "Todas":
        d = d[d["provincia"] == F["prov"]]; titulo = f"Provincia {F['prov']}"
    else:
        titulo = "Departamento de Puno"
    encabezado(F, d)
    if d.empty:
        print("Sin datos con estos filtros."); return
    f = pd.to_datetime(d["periodo"].astype(str) + "01", format="%Y%m%d")
    s = d.assign(fecha=f).groupby("fecha")[["consumo_kwh", "facturacion_soles", "suministros_mes"]].sum()
    s = s.reindex(pd.date_range(f.min(), f.max(), freq="MS"))
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, subplot_titles=("Consumo mensual (kWh)", "Facturación mensual (S/)"))
    fig.add_trace(go.Scatter(x=s.index, y=s["consumo_kwh"], mode="lines+markers", name="Consumo", line=dict(color=COLOR_CONSUMO), connectgaps=False,
                             customdata=s["suministros_mes"], hovertemplate="%{x|%Y-%m}<br>%{y:,.0f} kWh<br>Suministros: %{customdata:,.0f}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=s.index, y=s["facturacion_soles"], mode="lines+markers", name="Facturación", line=dict(color=COLOR_FACTURA), connectgaps=False,
                             hovertemplate="%{x|%Y-%m}<br>S/ %{y:,.0f}<extra></extra>"), row=2, col=1)
    fig.update_yaxes(tickformat=",.0f"); fig.update_xaxes(title_text="Mes", row=2, col=1)
    fig.update_layout(title=f"Ficha territorial: {titulo}", height=620, showlegend=False)
    fig.show()
    t = d.groupby("tarifa")[["suministros_mes", "consumo_kwh", "facturacion_soles"]].sum().sort_values("facturacion_soles", ascending=False)
    t["% facturación"] = 100 * t["facturacion_soles"] / t["facturacion_soles"].sum()
    fig2 = px.bar(t.reset_index(), x="tarifa", y="% facturación", color="tarifa", hover_data={"suministros_mes": ":,.0f", "consumo_kwh": ":,.0f", "facturacion_soles": ":,.0f"},
                  title=f"Mezcla tarifaria (participación en la facturación) — {titulo}", labels={"tarifa": "Tarifa", "% facturación": "% de la facturación"})
    fig2.update_layout(height=380, showlegend=False)
    fig2.show()
mostrar(vista_ficha)

Output()

## 10. Calidad y cobertura geográfica

¿Qué zonas tienen datos más completos o más problemáticos? Se resume por provincia: porcentaje de la facturación proveniente de tarifas **sin kWh informado** (en esas zonas el consumo está subestimado), porcentaje de suministros‑mes con kWh = 0 y cobertura de periodos.

In [13]:
tot_p = len(PERIODOS)
cal = (CUBO.assign(f_sin=np.where(CUBO["tarifa"].isin(TARIFAS_SIN_KWH), CUBO["facturacion_soles"], 0.0))
           .groupby("provincia")
           .agg(distritos=("zona", "nunique"), periodos=("periodo", "nunique"), suministros_mes=("suministros_mes", "sum"),
                kwh_cero=("suministros_kwh_cero", "sum"), fact=("facturacion_soles", "sum"), fact_sin=("f_sin", "sum")))
cal["% fact. de tarifas sin kWh informado"] = 100 * cal["fact_sin"] / cal["fact"]
cal["% suministros-mes con kWh = 0"] = 100 * cal["kwh_cero"] / cal["suministros_mes"]
cal["cobertura de periodos (%)"] = 100 * cal["periodos"] / tot_p
display(cal.sort_values("fact", ascending=False)[["distritos", "suministros_mes", "% suministros-mes con kWh = 0", "% fact. de tarifas sin kWh informado", "cobertura de periodos (%)"]]
        .style.format({"suministros_mes": "{:,.0f}", "% suministros-mes con kWh = 0": "{:.1f}", "% fact. de tarifas sin kWh informado": "{:.1f}", "cobertura de periodos (%)": "{:.0f}"}))

dist_per = CUBO.groupby("zona")["periodo"].nunique()
incompletos = dist_per[dist_per < tot_p]
print(f"Distritos con menos de {tot_p} periodos con datos: {len(incompletos)}")
if len(incompletos):
    display(pd.DataFrame({"distrito": incompletos.index.map(ETIQUETA), "periodos con datos": incompletos.values}).sort_values("periodos con datos").head(15))

s = cal.sort_values("% fact. de tarifas sin kWh informado")
fig = px.bar(s.reset_index(), y="provincia", x="% fact. de tarifas sin kWh informado", orientation="h", color_discrete_sequence=["#9467bd"],
             title="Provincias: % de la facturación que proviene de tarifas sin kWh informado", labels={"% fact. de tarifas sin kWh informado": "% de la facturación", "provincia": "Provincia"},
             hover_data={"suministros_mes": ":,.0f"})
fig.update_layout(height=470); fig.show()

,distritos,suministros_mes,% suministros-mes con kWh = 0,% fact. de tarifas sin kWh informado,cobertura de periodos (%)
provincia,,,,,
SAN ROMAN,5,"4,891,906",9.0,7.5,100
PUNO,15,"3,017,681",12.4,13.2,100
SAN ANTONIO DE PUTINA,5,"526,961",14.4,44.9,100
CARABAYA,10,"285,556",14.0,0.8,100
LAMPA,10,"427,713",20.8,76.2,100
MELGAR,9,"873,444",15.7,8.4,100
AZANGARO,15,"1,250,086",17.1,8.0,100
EL COLLAO,5,"886,674",19.7,10.4,100
CHUCUITO,7,"760,039",22.0,3.6,100


Distritos con menos de 41 periodos con datos: 0


### Sobre el mapa interactivo

Los archivos **no contienen coordenadas ni geometrías**; solo el código `ubigeo` (parcialmente inconsistente con los nombres) y nombres. Dibujar un mapa exigiría un archivo externo de límites distritales (por ejemplo, un shapefile/GeoJSON oficial del INEI). Como no hay una fuente confiable dentro del proyecto, **no se genera mapa ni se inventan ubicaciones**. Si dispone de ese archivo, bastaría unirlo por `ubigeo_modal` (columna de `GEO`) y usar `px.choropleth_map` con el cubo de la sección 2.

## 11. Hallazgos calculados (vista completa: todo el periodo, todas las tarifas)

Texto generado desde los números reales; use los filtros de arriba para explorar subconjuntos.

In [14]:
dd = CUBO.copy()
gp = agregar(dd, ["provincia"]); gd = agregar(dd, ["zona", "etiqueta", "provincia"])
for g_, c in [(gp, "consumo_kwh"), (gp, "facturacion_soles"), (gp, "suministros_mes")]:
    g_["part_" + c] = 100 * g_[c] / g_[c].sum()
# --- Sensibilidad: ranking por NOMBRE (provincia|distrito) vs. por UBIGEO registrado ---
su = q("""SELECT ubigeo, SUM(consumo_kwh) AS kwh FROM (SELECT ubigeo, consumo_kwh FROM clientes UNION ALL SELECT ubigeo, consumo_kwh FROM agregado) GROUP BY 1""")
mapa_ub = GEO.reset_index().drop_duplicates("ubigeo_modal").set_index("ubigeo_modal")["zona"]
su["zona"] = su["ubigeo"].map(mapa_ub)
su = su.dropna(subset=["zona"]).groupby("zona")["kwh"].sum()
cmp_ = gd.set_index("zona")["consumo_kwh"].to_frame("por_nombre").join(su.rename("por_ubigeo"), how="inner")
rho_s, _ = stats.spearmanr(cmp_["por_nombre"], cmp_["por_ubigeo"])
sol = len(set(cmp_["por_nombre"].nlargest(10).index) & set(cmp_["por_ubigeo"].nlargest(10).index))
H = []
t3c = gp.nlargest(3, "consumo_kwh"); t3f = gp.nlargest(3, "facturacion_soles")
H.append("PROVINCIAS por consumo: " + "; ".join(f"{r.provincia} {r.part_consumo_kwh:.1f}%" for r in t3c.itertuples()) + f" (suman {t3c['part_consumo_kwh'].sum():.1f}%).")
H.append("PROVINCIAS por facturación: " + "; ".join(f"{r.provincia} {r.part_facturacion_soles:.1f}%" for r in t3f.itertuples()) + f" (suman {t3f['part_facturacion_soles'].sum():.1f}%).")
rho_p, _ = stats.spearmanr(gp["consumo_kwh"], gp["facturacion_soles"]); rho_d, _ = stats.spearmanr(gd["consumo_kwh"], gd["facturacion_soles"])
H.append(f"COINCIDENCIA: Spearman consumo–facturación = {rho_p:.2f} entre provincias y {rho_d:.2f} entre distritos.")
n = 10
tc = gd.nlargest(n, "consumo_kwh"); tf = gd.nlargest(n, "facturacion_soles")
H.append(f"TOP {n}: {len(set(tc['zona']) & set(tf['zona']))} distritos coinciden en ambos rankings. Mayor consumo: {tc.iloc[0]['etiqueta']} ({100*tc.iloc[0]['consumo_kwh']/gd['consumo_kwh'].sum():.1f}%). Mayor facturación: {tf.iloc[0]['etiqueta']} ({100*tf.iloc[0]['facturacion_soles']/gd['facturacion_soles'].sum():.1f}%).")
gv = gd[(gd["suministros_mes_kwh"] >= 1000) & gd["kwh_por_suministro"].notna()]
if len(gv) >= 4:
    hi = gv.nlargest(3, "kwh_por_suministro"); lo = gv.nsmallest(3, "kwh_por_suministro")
    H.append("PROMEDIO kWh/suministro-mes (≥1 000 suministros-mes): mayores " + "; ".join(f"{r.etiqueta} {r.kwh_por_suministro:.1f}" for r in hi.itertuples())
             + " | menores " + "; ".join(f"{r.etiqueta} {r.kwh_por_suministro:.1f}" for r in lo.itertuples()) + ".")
    rho_a, _ = stats.spearmanr(gv["consumo_kwh"], gv["kwh_por_suministro"])
    H.append(f"TOTAL vs PROMEDIO: Spearman = {rho_a:.2f} entre consumo total y consumo promedio por suministro (distritos con ≥1 000 suministros-mes).")
for c, nombre in [("consumo_kwh", "consumo"), ("facturacion_soles", "facturación"), ("suministros_mes", "suministros")]:
    x = gd[c].clip(lower=0).values; sh = np.sort(x / x.sum())[::-1]
    H.append(f"CONCENTRACIÓN ({nombre}): Gini {gini(x):.2f}, HHI {10000*(sh**2).sum():,.0f}, 5 mayores distritos = {100*sh[:5].sum():.1f}%.")
cs = cal["% fact. de tarifas sin kWh informado"].sort_values(ascending=False)
H.append(f"CALIDAD: provincias con más facturación de tarifas sin kWh informado: " + "; ".join(f"{k} {v:.1f}%" for k, v in cs.head(3).items()) + ".")
H.append(f"SENSIBILIDAD DE LA CLAVE: Spearman entre consumo por nombre y por ubigeo = {rho_s:.3f}; {sol} de 10 distritos coinciden en el Top 10 (ubigeo contradice al nombre en {pct_disc:.2f}% de los suministros-mes).")
for h in H: print("•", h, "\n")

• PROVINCIAS por consumo: SAN ROMAN 40.8%; PUNO 22.8%; SAN ANTONIO DE PUTINA 7.4% (suman 71.1%). 

• PROVINCIAS por facturación: SAN ROMAN 36.8%; PUNO 21.6%; SAN ANTONIO DE PUTINA 10.4% (suman 68.9%). 

• COINCIDENCIA: Spearman consumo–facturación = 0.77 entre provincias y 0.91 entre distritos. 

• TOP 10: 8 distritos coinciden en ambos rankings. Mayor consumo: JULIACA (San Roman) (39.0%). Mayor facturación: JULIACA (San Roman) (35.1%). 

• PROMEDIO kWh/suministro-mes (≥1 000 suministros-mes): mayores SAN JUAN DEL ORO (Sandia) 618.5; CONIMA (Moho) 616.0; ITUATA (Carabaya) 519.3 | menores CONDURIRI (El Collao) 3.0; PISACOMA (Chucuito) 5.1; SAN JOSE (Azangaro) 5.8. 

• TOTAL vs PROMEDIO: Spearman = 0.67 entre consumo total y consumo promedio por suministro (distritos con ≥1 000 suministros-mes). 

• CONCENTRACIÓN (consumo): Gini 0.86, HHI 2,013, 5 mayores distritos = 70.8%. 

• CONCENTRACIÓN (facturación): Gini 0.86, HHI 1,755, 5 mayores distritos = 71.9%. 

• CONCENTRACIÓN (suministros)

## 12. Conclusiones

Las cifras están en la celda anterior. Cómo leerlas:

**1. ¿Qué descubrimos?** — Qué provincias y distritos concentran consumo y facturación (*PROVINCIAS*, *TOP*), si los rankings coinciden (*COINCIDENCIA*), y que **el total no predice el promedio por suministro** (*TOTAL vs PROMEDIO*).

**2. ¿Qué respalda el hallazgo?** — Barras por provincia (sec. 3), rankings de distritos (4), dispersión consumo–facturación (5), Pareto y treemap (6), total vs. promedio (7), Lorenz/Gini/HHI (8).

**3. ¿Por qué importa?** — Para priorizar zonas de atención comercial y de red: el volumen total orienta dónde está el grueso de la demanda; el promedio por suministro orienta qué zonas tienen clientes de mayor consumo (potencialmente comerciales/industriales) o cartera residencial de bajo consumo.

**4. Acciones/análisis recomendados**
* Priorizar la atención en las zonas que suman la mayor parte del consumo y la facturación (según Pareto/Gini).
* Revisar los distritos cuya posición por facturación difiere mucho de su posición por consumo (mezcla tarifaria o tarifas sin kWh).
* Obtener un archivo oficial de límites distritales si se desea un mapa, y pedir a la fuente que corrija las filas donde `ubigeo` contradice al nombre.
* Incorporar el kWh de las tarifas que hoy figuran con cero para estimar la demanda real por zona.

**5. Limitaciones**
* Consumo y facturación están **subestimados/incompletos** donde predominan tarifas sin kWh informado.
* 2026 es agregado (se usa solo para totales y conteos); `codigo_cliente` no es identificador estable y el `ubigeo` contradice al nombre en una pequeña fracción de filas, por eso no se siguen suministros individuales.
* El promedio por suministro‑mes depende de la mezcla de tarifas y de la cobertura de periodos.
* Un total mayor **no** significa un consumo por cliente mayor.

## 13. Mapas de Puno con GeoJSON y principios de Gestalt

Hasta aquí los gráficos comparan territorios con barras y dispersiones. En esta sección el **territorio se dibuja**: se une el GeoJSON de distritos (`peru_distrital_simple.geojson`) con el cubo de la sección 2 y se construyen mapas que reaccionan al **mismo panel de filtros** (periodo, provincia, tarifa, métrica).

**No se necesitan librerías nuevas.** Se usan `plotly.graph_objects` (`Choropleth` y `Scattergeo`), `json` y la biblioteca estándar. Los mapas se dibujan **sin mapa base** (no descargan *tiles*), así que funcionan sin internet y el fondo queda limpio. Los límites provinciales se obtienen disolviendo los distritos con Python puro: el GeoJSON conserva los vértices compartidos entre distritos vecinos, de modo que las aristas que pertenecen a un solo distrito de la provincia son su frontera.

**Principios de la Gestalt aplicados**

| Principio | Idea | Cómo se aplica en estos gráficos |
|---|---|---|
| **Figura–fondo** | Separar lo importante de lo que acompaña | Sin mapa base; distritos sin datos en gris claro (fondo); la provincia elegida en el filtro con contorno **naranja** (el mismo resaltado de la sección 3) |
| **Similitud** | Lo parecido se percibe como grupo | Misma clase = mismo color en el mapa y en las barras. Colores con significado fijo en todo el notebook: **verde = consumo**, **rojo = facturación**, **azul = suministros** |
| **Proximidad** | Lo cercano se lee como unidad | Ranking de barras pegado al mapa y leyenda justo debajo; etiquetas sobre la propia provincia (sin leyenda lejana) |
| **Cierre** | El ojo completa formas cerradas | Cada provincia tiene contorno cerrado; el fondo del departamento no necesita marco |
| **Región común** | Un borde agrupa lo que contiene | Contorno provincial grueso sobre líneas distritales finas: se ve la jerarquía provincia → distrito |
| **Continuidad / conexión** | El ojo sigue líneas | Gráfico de mancuernas: una línea une participación en consumo y en facturación de cada provincia |
| **Simplicidad (Prägnanz)** | Preferir la forma más simple | 5 clases por cuantiles en lugar de un degradado continuo; sin ejes, cuadrícula ni coordenadas |
| **Punto focal** | Lo distinto atrae la mirada | El color más oscuro y la etiqueta en negrita marcan a la provincia/distrito líder |
| **Simetría y destino común** | Lo alineado y lo que cambia a la vez se agrupa | Tres mapas idénticos con **una sola escala**; todos se actualizan juntos con el panel de filtros |

**Cómo leerlos.** Un color más oscuro significa más consumo / facturación / suministros. Pase el cursor sobre un distrito para ver los valores de su provincia. Para el mapa distrital use el botón **Métrica** del panel (el rótulo dice «sec. 6», pero aplica también aquí).

**Limitación.** El cruce con el GeoJSON se hace por nombre (provincia, distrito) —la misma clave que decidimos en la sección 1— y, de respaldo, por `ubigeo_modal`. La siguiente celda informa qué cruzó y qué no. Un distrito sin polígono sigue sumando en los totales provinciales (los valores salen de los datos, no de la geometría), pero no se ve en el mapa.

In [15]:
# ============================================================
# 13.1 PREPARACIÓN GEOESPACIAL
# Qué hace: lee el GeoJSON, lo cruza con los distritos del notebook,
# informa qué cruzó y calcula límites provinciales y centroides
# (solo con biblioteca estándar: sin geopandas ni shapely).
# ============================================================
import unicodedata, re, collections

GEOJSON_RUTA = None      # ← opcional: ruta explícita al GeoJSON si el notebook no lo encuentra solo
NOMBRE_GEOJSON = "peru_distrital_simple.geojson"

def localizar_geojson():
    """Busca el GeoJSON en la raíz del proyecto y alrededores."""
    if GEOJSON_RUTA:
        ruta = Path(GEOJSON_RUTA)
        if ruta.exists():
            return ruta
        raise FileNotFoundError(f"No existe la ruta indicada en GEOJSON_RUTA: {ruta}")
    for c in [RAIZ / NOMBRE_GEOJSON, SILVER / NOMBRE_GEOJSON, SILVER.parent / NOMBRE_GEOJSON, Path.cwd() / NOMBRE_GEOJSON]:
        if c.exists():
            return c
    for c in RAIZ.rglob(NOMBRE_GEOJSON):
        return c
    raise FileNotFoundError(f"No encuentro '{NOMBRE_GEOJSON}' dentro de {RAIZ}. Cópialo al proyecto o asigna su ruta a GEOJSON_RUTA.")

def _norm(s):
    """Normaliza nombres: mayúsculas, sin tildes ni signos, espacios simples (AZÁNGARO → AZANGARO)."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii").upper()
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9]+", " ", s)).strip()

def _ubigeo(x):
    s = str(x).strip()
    s = s[:-2] if s.endswith(".0") else s
    return s.zfill(6)

# ---------- 1) Leer el GeoJSON y quedarnos con Puno ----------
RUTA_GEOJSON = localizar_geojson()
with open(RUTA_GEOJSON, encoding="utf-8") as fh:
    _gj = json.load(fh)
_puno = [f for f in _gj["features"]
         if str(f["properties"].get("IDDPTO")) == "21" or _norm(f["properties"].get("NOMBDEP")) == "PUNO"]
if not _puno:
    raise ValueError("El GeoJSON no contiene distritos de Puno (se buscó IDDPTO = '21' o NOMBDEP = 'PUNO').")
_faltan = {"IDDIST", "IDPROV", "NOMBDIST", "NOMBPROV"} - set(_puno[0]["properties"])
if _faltan:
    raise KeyError(f"Al GeoJSON le faltan las propiedades {sorted(_faltan)}; revise el archivo.")
_ids_geo = {f["properties"]["IDDIST"] for f in _puno}
_idx_nombre = {(_norm(f["properties"]["NOMBPROV"]), _norm(f["properties"]["NOMBDIST"])): f["properties"]["IDDIST"] for f in _puno}

# ---------- 2) Cruce con GEO: primero por nombre (provincia, distrito); de respaldo por ubigeo ----------
ZONA2ID, _sin_cruce, _por = {}, [], collections.Counter()
for zona, r in GEO.iterrows():
    k = (_norm(r["provincia"]), _norm(r["distrito"]))
    if k in _idx_nombre:
        ZONA2ID[zona] = _idx_nombre[k]; _por["nombre"] += 1
    elif _ubigeo(r["ubigeo_modal"]) in _ids_geo:
        ZONA2ID[zona] = _ubigeo(r["ubigeo_modal"]); _por["ubigeo"] += 1
    else:
        _sin_cruce.append(zona)
ID2ZONA = {}
_dup = []
for z, i in ZONA2ID.items():
    if i in ID2ZONA:
        _dup.append((ID2ZONA[i], z))
    ID2ZONA[i] = z

PROV_POR_NOMBRE = {_norm(p): p for p in PROVINCIAS}

# ---------- 3) Diccionario de distritos dibujables y GeoJSON mínimo (solo id + geometría) ----------
DISTRITOS, FEAT = {}, {}
for f in _puno:
    if not f["geometry"]:
        continue
    p = f["properties"]; i = p["IDDIST"]; zona = ID2ZONA.get(i)
    prov = GEO.at[zona, "provincia"] if zona else PROV_POR_NOMBRE.get(_norm(p["NOMBPROV"]))
    DISTRITOS[i] = {"zona": zona, "prov": prov, "distrito": p["NOMBDIST"].title(), "idprov": p["IDPROV"]}
    FEAT[i] = f
GJ_PUNO = {"type": "FeatureCollection",
           "features": [{"type": "Feature", "id": i, "properties": {"IDDIST": i}, "geometry": f["geometry"]} for i, f in FEAT.items()]}

# provincia (nombre en los datos) → código de provincia del GeoJSON (el más frecuente entre sus distritos)
_c = collections.defaultdict(collections.Counter)
for i, info in DISTRITOS.items():
    if info["prov"]:
        _c[info["prov"]][info["idprov"]] += 1
PROV_ID = {p: cnt.most_common(1)[0][0] for p, cnt in _c.items()}

# ---------- 4) Límites provinciales: aristas que pertenecen a UN solo distrito del grupo ----------
def _anillos(geom):
    polis = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
    for poli in polis:
        for anillo in poli:
            yield anillo

def _pt(p):
    return (round(p[0], 5), round(p[1], 5))

def bordes_por_grupo(features, clave):
    """Devuelve {grupo: (lon, lat)} con segmentos separados por None, listos para Scattergeo."""
    cont = collections.defaultdict(collections.Counter)
    for f in features:
        if not f["geometry"]:
            continue
        g = clave(f)
        for anillo in _anillos(f["geometry"]):
            for a, b in zip(anillo[:-1], anillo[1:]):
                a, b = _pt(a), _pt(b)
                if a != b:
                    cont[g][(a, b) if a < b else (b, a)] += 1
    salida = {}
    for g, c in cont.items():
        xs, ys = [], []
        for (a, b), n in c.items():
            if n == 1:
                xs += [a[0], b[0], None]; ys += [a[1], b[1], None]
        salida[g] = (xs, ys)
    return salida

BORDES_PROV = bordes_por_grupo(FEAT.values(), lambda f: f["properties"]["IDPROV"])

def centroide(features):
    """Centroide ponderado por área (fórmula del polígono), para ubicar etiquetas."""
    A = cx = cy = 0.0
    for f in features:
        for poli in ([f["geometry"]["coordinates"]] if f["geometry"]["type"] == "Polygon" else f["geometry"]["coordinates"]):
            for k, anillo in enumerate(poli):
                a2 = sx = sy = 0.0
                for p0, p1 in zip(anillo[:-1], anillo[1:]):
                    cr = p0[0] * p1[1] - p1[0] * p0[1]
                    a2 += cr; sx += (p0[0] + p1[0]) * cr; sy += (p0[1] + p1[1]) * cr
                if a2 == 0:
                    continue
                area = a2 / 2
                w = abs(area) * (1 if k == 0 else -1)
                A += w; cx += sx / (6 * area) * w; cy += sy / (6 * area) * w
    return (cx / A, cy / A) if A else None

CENT_DIST = {i: centroide([f]) for i, f in FEAT.items()}
CENT_PROV = {}
for p in PROV_ID:
    c = centroide([FEAT[i] for i, info in DISTRITOS.items() if info["prov"] == p])
    if c:
        CENT_PROV[p] = c

# ---------- 5) Informe del cruce ----------
print(f"GeoJSON: {RUTA_GEOJSON.name} | distritos de Puno: {len(_puno)} (con geometría: {len(DISTRITOS)})")
print(f"Cruce con los datos → por nombre: {_por['nombre']} | por ubigeo (respaldo): {_por['ubigeo']} | sin cruzar: {len(_sin_cruce)} de {len(GEO)}")
if _dup:
    print("⚠ Dos distritos de los datos cayeron en el mismo polígono:", _dup)
sin_poligono = [z for z in GEO.index if ZONA2ID.get(z) not in DISTRITOS]
if sin_poligono:
    tot_z = CUBO.groupby("zona")[["consumo_kwh", "facturacion_soles"]].sum()
    s_p = tot_z.loc[tot_z.index.intersection(sin_poligono)]
    print(f"⚠ {len(sin_poligono)} distrito(s) de los datos NO se ven en el mapa (sin polígono): "
          f"{100 * s_p['consumo_kwh'].sum() / tot_z['consumo_kwh'].sum():.2f}% del consumo y "
          f"{100 * s_p['facturacion_soles'].sum() / tot_z['facturacion_soles'].sum():.2f}% de la facturación. "
          "Sus valores sí suman en los totales provinciales.")
    display(pd.DataFrame({"zona sin polígono": [ETIQUETA.get(z, z) for z in sin_poligono]}).head(15))
fuera_geo = sorted(i for i in _ids_geo if i not in ID2ZONA)
if fuera_geo:
    print(f"ℹ {len(fuera_geo)} distrito(s) del GeoJSON no tienen datos de consumo (se dibujan en gris).")
falt_prov = [p for p in PROVINCIAS if p not in PROV_ID]
print("Provincias de los datos sin equivalente en el GeoJSON:", falt_prov or "ninguna")
print(f"Límites provinciales calculados para {len(BORDES_PROV)} provincias.")

GeoJSON: peru_distrital_simple.geojson | distritos de Puno: 109 (con geometría: 108)
Cruce con los datos → por nombre: 106 | por ubigeo (respaldo): 2 | sin cruzar: 1 de 109
⚠ 2 distrito(s) de los datos NO se ven en el mapa (sin polígono): 0.94% del consumo y 0.83% de la facturación. Sus valores sí suman en los totales provinciales.


,zona sin polígono
0,AMANTANI (Puno)
1,SAN MIGUEL (San Roman)


ℹ 1 distrito(s) del GeoJSON no tienen datos de consumo (se dibujan en gris).
Provincias de los datos sin equivalente en el GeoJSON: ninguna
Límites provinciales calculados para 13 provincias.


### 13.2 Herramientas comunes de los mapas

**Qué hace:** define paletas, la clasificación en 5 clases por cuantiles y las capas reutilizables (polígonos, contornos, etiquetas). Así **todos los mapas comparten el mismo lenguaje visual**, que es la base de la similitud y la simetría.
**Por qué clases y no degradado continuo:** con pocas provincias muy desiguales, un degradado lineal deja casi todo pálido y una o dos zonas oscuras. Las clases por cuantiles agrupan visualmente y permiten comparar de un vistazo.

In [16]:
# ============================================================
# 13.2 HERRAMIENTAS COMUNES DE LOS MAPAS
# ============================================================
PAL_CONSUMO = ["#edf8e9", "#bae4b3", "#74c476", "#31a354", "#006d2c"]   # verde  = consumo
PAL_FACTURA = ["#fee5d9", "#fcae91", "#fb6a4a", "#de2d26", "#a50f15"]   # rojo   = facturación
PAL_SUMIN   = ["#eff3ff", "#bdd7e7", "#6baed6", "#3182bd", "#08519c"]   # azul   = suministros
COLOR_GRIS_FONDO = "#e3e3e3"      # distritos sin datos: quedan en el "fondo"
COLOR_SELECCION  = "#ff7f0e"      # provincia elegida en el filtro (mismo naranja de la sección 3)
COLOR_CONTORNO   = "#2b2b2b"

def fmtv(x):
    return f"{x:,.0f}" if abs(x) >= 100 else f"{x:,.1f}"

def _texto_sobre(hex_color):
    """Texto blanco sobre colores oscuros y gris oscuro sobre colores claros."""
    h = hex_color.lstrip("#"); r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return "white" if 0.299 * r + 0.587 * g + 0.114 * b < 140 else "#222222"

def clasificar(serie, paleta, factor=1.0):
    """Agrupa los valores en hasta 5 clases por cuantiles. Devuelve clase (0..m-1), colores y etiquetas."""
    v = pd.Series(serie, dtype=float).dropna()
    cortes = np.unique(np.quantile(v.values, np.linspace(0, 1, len(paleta) + 1)))
    if len(cortes) < 2:
        cortes = np.array([cortes[0], cortes[0]])
    m = len(cortes) - 1
    clase = pd.Series(np.digitize(v.values, cortes[1:-1]) if m > 1 else np.zeros(len(v), dtype=int), index=v.index)
    pos = [round(j * (len(paleta) - 1) / (m - 1)) for j in range(m)] if m > 1 else [len(paleta) // 2]
    etiquetas = [f"{fmtv(cortes[j] / factor)} – {fmtv(cortes[j + 1] / factor)}" for j in range(m)]
    return {"clase": clase, "m": m, "colores": [paleta[i] for i in pos], "etiquetas": etiquetas}

def escala_discreta(colores):
    """Escala 'escalonada' de Plotly: cada clase es un color plano."""
    m = len(colores); esc = []
    for j, c in enumerate(colores):
        esc += [[j / m, c], [(j + 1) / m, c]]
    return esc

def barra_color(cl, orientacion="h", titulo=""):
    """Leyenda horizontal bajo el mapa (proximidad): una marca por clase con su rango."""
    return dict(orientation=orientacion, x=0.31, xanchor="center", y=0.115, yanchor="top", len=0.58, thickness=12,
                title=dict(text=titulo, side="top"), tickvals=[j + 0.5 for j in range(cl["m"])], ticktext=cl["etiquetas"],
                tickfont=dict(size=10))

def por_distrito_desde_provincia(z_prov, hover_prov):
    """Reparte el valor de cada provincia entre sus distritos (todos de una provincia comparten color)."""
    z, h = {}, {}
    for i, info in DISTRITOS.items():
        p = info["prov"]
        if p in z_prov:
            z[i] = z_prov[p]
            h[i] = f"<b>{info['distrito']}</b> · prov. {p.title()}<br>{hover_prov[p]}"
    return z, h

def capa_poligonos(z, hover, colorscale=None, zmin=None, zmax=None, colorbar=None, ids=None, geo="geo", coloraxis=None):
    """Capa de distritos: con datos → color de su clase; sin datos → gris de fondo."""
    ids = list(DISTRITOS) if ids is None else list(ids)
    con = [i for i in ids if i in z]; sin = [i for i in ids if i not in z]
    trazas = []
    if sin:
        trazas.append(go.Choropleth(
            geojson=GJ_PUNO, featureidkey="properties.IDDIST", locations=sin, z=[0] * len(sin),
            colorscale=[[0, COLOR_GRIS_FONDO], [1, COLOR_GRIS_FONDO]], zmin=0, zmax=1, showscale=False,
            marker=dict(line=dict(color="white", width=0.4)),
            text=[f"<b>{DISTRITOS[i]['distrito']}</b><br>sin datos con estos filtros" for i in sin],
            hovertemplate="%{text}<extra></extra>", geo=geo))
    if con:
        d = dict(geojson=GJ_PUNO, featureidkey="properties.IDDIST", locations=con, z=[z[i] for i in con],
                 text=[hover[i] for i in con], hovertemplate="%{text}<extra></extra>",
                 marker=dict(line=dict(color="rgba(255,255,255,0.45)", width=0.3)), geo=geo)
        if coloraxis:
            d["coloraxis"] = coloraxis
        else:
            d.update(colorscale=colorscale, zmin=zmin, zmax=zmax, colorbar=colorbar, showscale=colorbar is not None)
        trazas.append(go.Choropleth(**d))
    return trazas

def capa_bordes(idprovs, color=COLOR_CONTORNO, ancho=1.5, geo="geo"):
    """Contorno cerrado de las provincias indicadas (región común / cierre)."""
    xs, ys = [], []
    for i in idprovs:
        if i in BORDES_PROV:
            xs += BORDES_PROV[i][0]; ys += BORDES_PROV[i][1]
    return go.Scattergeo(lon=xs, lat=ys, mode="lines", line=dict(color=color, width=ancho), hoverinfo="skip", showlegend=False, geo=geo)

def capa_etiquetas(textos, colores, centros, tam=10, geo="geo"):
    """Etiquetas de texto ubicadas en el centroide de cada zona."""
    claves = [k for k in textos if centros.get(k)]
    return go.Scattergeo(lon=[centros[k][0] for k in claves], lat=[centros[k][1] for k in claves], mode="text",
                         text=[textos[k] for k in claves], textfont=dict(size=tam, color=[colores[k] for k in claves]),
                         hoverinfo="skip", showlegend=False, geo=geo)

def ajustar_geo(fig, clave="geo", x=(0, 1), y=(0, 1)):
    """Encuadra el mapa en los polígonos dibujados, sin mapa base ni coordenadas (figura–fondo, simplicidad)."""
    fig.update_layout(**{clave: dict(domain=dict(x=list(x), y=list(y)), fitbounds="locations", visible=False,
                                     projection=dict(type="mercator"), bgcolor="rgba(0,0,0,0)")})

def tabla_provincias(F):
    """Totales por provincia SIN filtrar por provincia: el mapa muestra todas y resalta la elegida."""
    d = preparar(F)
    return d, agregar(d, ["provincia"]).set_index("provincia")

def layout_mapa_barras(fig, categorias, titulo, subtitulo, titulo_x, zeroline=False):
    """Mapa a la izquierda y ranking a la derecha, en una sola figura (proximidad)."""
    ajustar_geo(fig, "geo", x=(0.0, 0.62), y=(0.13, 1.0))
    fig.update_layout(title=dict(text=f"<b>{titulo}</b><br><sup>{subtitulo}</sup>", x=0.01),
                      height=680, margin=dict(l=10, r=50, t=95, b=110), plot_bgcolor="white",
                      xaxis=dict(domain=[0.75, 0.96], anchor="y", title=titulo_x, showgrid=False, zeroline=zeroline, zerolinecolor="#888"),
                      yaxis=dict(domain=[0.13, 0.93], anchor="x", type="category", categoryorder="array",
                                 categoryarray=categorias, showgrid=False))
    fig.add_annotation(text="<b>Ranking</b>", x=0.855, y=0.985, xref="paper", yref="paper", showarrow=False)

def subtitulo_mapa(F):
    s = f"Periodos {F['p0']}–{F['p1']} · {len(F['tarifas'])} de {len(TARIFAS)} tarifas"
    if F["prov"] != "Todas":
        s += " · contorno naranja = provincia elegida en el filtro"
    return s

def mapa_provincial(F, col, nombre, unidad, factor, paleta, titulo):
    """Mapa coroplético por provincia + ranking de barras con los mismos colores."""
    d, g = tabla_provincias(F)
    s = g[g[col] > 0]
    if s.empty:
        print("Sin datos con estos filtros."); return
    cl = clasificar(s[col], paleta, factor)
    clase, m = cl["clase"], cl["m"]
    part = 100 * s[col] / s[col].sum()
    color_de = {p: cl["colores"][clase[p]] for p in s.index}
    hover_prov = {p: (f"Consumo: {s.at[p, 'consumo_kwh']:,.0f} kWh<br>Facturación: S/ {s.at[p, 'facturacion_soles']:,.0f}"
                      f"<br>Participación en {nombre.lower()}: {part[p]:.1f}%<br>Suministros-mes: {s.at[p, 'suministros_mes']:,.0f}")
                  for p in s.index}
    z, h = por_distrito_desde_provincia({p: clase[p] + 0.5 for p in s.index}, hover_prov)

    fig = go.Figure()
    for t in capa_poligonos(z, h, escala_discreta(cl["colores"]), 0, m, barra_color(cl, titulo=unidad)):
        fig.add_trace(t)
    fig.add_trace(capa_bordes(PROV_ID.values(), COLOR_CONTORNO, 1.4))
    if F["prov"] in PROV_ID:
        fig.add_trace(capa_bordes([PROV_ID[F["prov"]]], COLOR_SELECCION, 4))
    lider = s[col].idxmax()
    fig.add_trace(capa_etiquetas({p: (f"<b>{p.title()}</b>" if p == lider else p.title()) for p in s.index},
                                 {p: _texto_sobre(color_de[p]) for p in s.index}, CENT_PROV))

    b = s[col].sort_values()
    sel = [p == F["prov"] for p in b.index]
    fig.add_trace(go.Bar(x=b.values / factor, y=[p.title() for p in b.index], orientation="h", showlegend=False,
                         marker=dict(color=[color_de[p] for p in b.index],
                                     line=dict(color=[COLOR_SELECCION if x else "rgba(0,0,0,0.25)" for x in sel], width=[3 if x else 0.6 for x in sel])),
                         text=[f"{part[p]:.1f}%" for p in b.index], textposition="outside", cliponaxis=False,
                         hovertemplate="<b>%{y}</b><br>%{x:,.1f} " + unidad + "<extra></extra>", xaxis="x", yaxis="y"))
    layout_mapa_barras(fig, [p.title() for p in b.index], titulo, subtitulo_mapa(F), unidad)
    fig.show()
    top3 = s[col].nlargest(3)
    print(f"{lider.title()} lidera con {part[lider]:.1f}% de {nombre.lower()}; las 3 primeras provincias suman {100 * top3.sum() / s[col].sum():.1f}%.")

### 13.3 Mapa de Puno: consumo por provincia

**Indicador:** consumo total (GWh = millones de kWh) por provincia, agrupado en 5 clases de **verde** (el color del consumo en todo el notebook).
**Gestalt:** *figura–fondo* (sin mapa base, gris para lo que no tiene datos, naranja para la provincia elegida), *similitud* (la misma clase tiene el mismo color en mapa y barras), *proximidad* (ranking y leyenda pegados al mapa), *cierre/región común* (contorno provincial grueso sobre líneas distritales finas) y *punto focal* (la provincia líder va en negrita).
**Cómo leerlo:** cuanto más oscuro el verde, más consumo. Las barras confirman el orden y muestran la participación %.
**Limitación:** el consumo de las tarifas sin kWh informado es cero, así que está subestimado donde esas tarifas pesan (ver sección 10).

In [17]:
def vista_mapa_consumo(**kw):
    F = leer(kw); d = preparar(F); encabezado(F, d)
    mapa_provincial(F, "consumo_kwh", "Consumo", "GWh (millones de kWh)", 1e6, PAL_CONSUMO, "Mapa de Puno · consumo de energía por provincia")
mostrar(vista_mapa_consumo)

Output()

### 13.4 Mapa coroplético: facturación por provincia

**Indicador:** facturación total (millones de S/) por provincia, en 5 clases de **rojo** (el color de la facturación en todo el notebook).
**Gestalt:** el mapa es **idéntico en forma y disposición** al anterior; solo cambia el color. Esa *simetría* y la *similitud* cromática permiten compararlos de memoria: basta alternar entre ambos para ver si las provincias oscuras son las mismas.
**Cómo leerlo:** rojo más oscuro = más soles facturados. Si una provincia cambia de clase respecto al mapa de consumo, la sección 13.5 muestra por cuánto.

In [19]:
def vista_mapa_facturacion(**kw):
    F = leer(kw); d = preparar(F); encabezado(F, d)
    mapa_provincial(F, "facturacion_soles", "Facturación", "Millones de S/", 1e6, PAL_FACTURA, "Mapa de Puno · facturación por provincia")
mostrar(vista_mapa_facturacion)

Output()

### 13.5 ¿Dónde no coinciden consumo y facturación? (mapa de diferencias)

**Indicador:** *participación en la facturación − participación en el consumo*, en puntos porcentuales (pp), por provincia. Responde la pregunta 3 sobre el territorio.
**Gestalt:** escala **divergente** con el centro en gris neutro (*figura–fondo*: lo que coincide se desvanece y lo que difiere salta a la vista). Verde = pesa más en consumo que en facturación; rojo = pesa más en facturación que en consumo (los mismos colores de las secciones anteriores). Las barras divergentes comparten el color de cada provincia (*similitud*).
**Cómo leerlo:** un rojo intenso indica una provincia que factura mucho más de lo que su consumo medido sugiere (mezcla tarifaria, cargos fijos o tarifas sin kWh informado).

In [20]:
def vista_mapa_diferencia(**kw):
    F = leer(kw); d, g = tabla_provincias(F); encabezado(F, d)
    s = g[(g["consumo_kwh"] > 0) & (g["facturacion_soles"] > 0)]
    if len(s) < 2:
        print("Pocas provincias con datos para esta combinación de filtros."); return
    pc = 100 * s["consumo_kwh"] / s["consumo_kwh"].sum(); pf = 100 * s["facturacion_soles"] / s["facturacion_soles"].sum()
    dif = pf - pc; lim = max(float(dif.abs().max()), 0.5)
    hover_prov = {p: (f"Participación en consumo: {pc[p]:.1f}%<br>Participación en facturación: {pf[p]:.1f}%<br>"
                      f"<b>Diferencia: {dif[p]:+.1f} pp</b>") for p in s.index}
    z, h = por_distrito_desde_provincia({p: float(dif[p]) for p in s.index}, hover_prov)
    esc = [[0, "#1b7837"], [0.25, "#a6dba0"], [0.5, "#f2f2f2"], [0.75, "#f4a582"], [1, "#b2182b"]]
    cb = dict(orientation="h", x=0.31, xanchor="center", y=0.115, yanchor="top", len=0.58, thickness=12, ticksuffix=" pp",
              title=dict(text="← pesa más en consumo   |   pesa más en facturación →", side="top"), tickfont=dict(size=10))
    fig = go.Figure()
    for t in capa_poligonos(z, h, esc, -lim, lim, cb):
        fig.add_trace(t)
    fig.add_trace(capa_bordes(PROV_ID.values(), COLOR_CONTORNO, 1.4))
    if F["prov"] in PROV_ID:
        fig.add_trace(capa_bordes([PROV_ID[F["prov"]]], COLOR_SELECCION, 4))
    foco = dif.abs().nlargest(2).index
    fig.add_trace(capa_etiquetas({p: (f"<b>{p.title()}</b><br>{dif[p]:+.1f} pp" if p in foco else p.title()) for p in s.index},
                                 {p: "#222222" for p in s.index}, CENT_PROV))
    b = dif.sort_values()
    sel = [p == F["prov"] for p in b.index]
    fig.add_trace(go.Bar(x=b.values, y=[p.title() for p in b.index], orientation="h", showlegend=False,
                         marker=dict(color=[COLOR_FACTURA if v > 0 else COLOR_CONSUMO for v in b.values],
                                     line=dict(color=[COLOR_SELECCION if x else "rgba(0,0,0,0.25)" for x in sel], width=[3 if x else 0.6 for x in sel])),
                         text=[f"{v:+.1f}" for v in b.values], textposition="outside", cliponaxis=False,
                         hovertemplate="<b>%{y}</b><br>%{x:+.2f} pp<extra></extra>", xaxis="x", yaxis="y"))
    layout_mapa_barras(fig, [p.title() for p in b.index], "Mapa de diferencias · participación en facturación − participación en consumo",
                       subtitulo_mapa(F), "puntos porcentuales", zeroline=True)
    fig.show()
    sube = dif.nlargest(1); baja = dif.nsmallest(1)
    print(f"Factura por encima de su consumo: {sube.index[0].title()} ({sube.iloc[0]:+.1f} pp). "
          f"Consume por encima de lo que factura: {baja.index[0].title()} ({baja.iloc[0]:+.1f} pp).")
    print("Recuerde: las tarifas sin kWh informado aportan facturación pero no consumo, y empujan hacia el rojo.")
mostrar(vista_mapa_diferencia)

Output()

### 13.6 Mapa distrital con contorno provincial

**Indicador:** la métrica elegida en el botón **Métrica** (consumo, facturación o suministros‑mes) por **distrito**, en 5 clases por cuantiles. Si elige una provincia en el filtro, el mapa se acerca a ella.
**Gestalt:** *región común* (contorno provincial grueso sobre los distritos), *figura–fondo* (gris = sin datos) y *punto focal* (se rotulan los 8 distritos con mayor valor).
**Cómo leerlo:** busque manchas de color oscuro agrupadas (*proximidad + similitud*): indican corredores de alta demanda. Un distrito aislado y oscuro dentro de una provincia clara es un caso a revisar.

In [21]:
def vista_mapa_distritos(**kw):
    F = leer(kw); d = preparar(F, aplicar_provincia=True); encabezado(F, d)
    col = F["metrica"]
    nombre, factor, unidad, paleta = {"consumo_kwh": ("Consumo", 1e6, "GWh", PAL_CONSUMO),
                                      "facturacion_soles": ("Facturación", 1e6, "Millones de S/", PAL_FACTURA),
                                      "suministros_mes": ("Suministros-mes", 1e3, "Miles de suministros-mes", PAL_SUMIN)}[col]
    g = agregar(d, ["zona"]); g["iddist"] = g["zona"].map(ZONA2ID)
    g = g[(g[col] > 0) & g["iddist"].isin(DISTRITOS.keys())]
    if g.empty:
        print("Sin distritos con datos y polígono para esta combinación de filtros."); return
    s = g.set_index("iddist"); total = s[col].sum()
    cl = clasificar(s[col], paleta, factor)
    z = {i: cl["clase"][i] + 0.5 for i in s.index}
    h = {i: (f"<b>{DISTRITOS[i]['distrito']}</b> ({DISTRITOS[i]['prov'].title() if DISTRITOS[i]['prov'] else '—'})<br>"
             f"Consumo: {s.at[i, 'consumo_kwh']:,.0f} kWh<br>Facturación: S/ {s.at[i, 'facturacion_soles']:,.0f}<br>"
             f"Suministros-mes: {s.at[i, 'suministros_mes']:,.0f}<br>Participación en {nombre.lower()} (del filtro): {100 * s.at[i, col] / total:.2f}%") for i in s.index}
    ids = [i for i, info in DISTRITOS.items() if F["prov"] == "Todas" or info["prov"] == F["prov"]]
    cb = barra_color(cl, titulo=unidad); cb["x"] = 0.5
    fig = go.Figure()
    for t in capa_poligonos(z, h, escala_discreta(cl["colores"]), 0, cl["m"], cb, ids=ids):
        fig.add_trace(t)
    if F["prov"] in PROV_ID:
        fig.add_trace(capa_bordes([PROV_ID[F["prov"]]], COLOR_SELECCION, 3.5))
    else:
        fig.add_trace(capa_bordes(PROV_ID.values(), COLOR_CONTORNO, 1.8))
    top = s[col].nlargest(min(8, len(s)))
    fig.add_trace(capa_etiquetas({i: DISTRITOS[i]["distrito"] for i in top.index},
                                 {i: _texto_sobre(cl["colores"][cl["clase"][i]]) for i in top.index}, CENT_DIST, tam=9))
    ajustar_geo(fig, "geo", x=(0, 1), y=(0.13, 1))
    fig.update_layout(title=dict(text=f"<b>Mapa distrital · {nombre}</b><br><sup>{subtitulo_mapa(F)} · contorno oscuro = provincia</sup>", x=0.01),
                      height=720, margin=dict(l=10, r=10, t=95, b=110))
    fig.show()
    print(f"Los 8 distritos rotulados suman {100 * top.sum() / total:.1f}% de {nombre.lower()} del filtro.")
mostrar(vista_mapa_distritos)

Output()

### 13.7 Tres mapas, una sola escala (consumo · facturación · suministros)

**Indicador:** participación (%) de cada provincia en cada una de las tres variables.
**Gestalt:** *simetría* (tres paneles del mismo tamaño y orden), *similitud* (**una única escala de color** para los tres: el mismo tono significa el mismo porcentaje en cualquier panel) y *destino común* (los tres cambian juntos con el filtro). Se rotulan solo las 3 provincias principales de cada panel para dejar un *punto focal*.
**Cómo leerlo:** si el patrón de manchas oscuras se repite, el territorio es coherente en las tres variables; si cambia, hay diferencias de precio, tarifa o tamaño de cartera.

In [22]:
def vista_mapas_paralelos(**kw):
    F = leer(kw); d, g = tabla_provincias(F); encabezado(F, d)
    mets = [("consumo_kwh", "Consumo (kWh)"), ("facturacion_soles", "Facturación (S/)"), ("suministros_mes", "Suministros-mes")]
    g = g[(g[[c for c, _ in mets]] > 0).all(axis=1)]
    if g.empty:
        print("Sin datos con estos filtros."); return
    part = {c: 100 * g[c] / g[c].sum() for c, _ in mets}
    vmax = float(max(p.max() for p in part.values()))
    doms = [(0.00, 0.31), (0.325, 0.635), (0.65, 0.96)]
    fig = go.Figure()
    for k, (c, nom) in enumerate(mets):
        clave = "geo" if k == 0 else f"geo{k + 1}"
        hov = {p: f"<b>{p.title()}</b><br>{nom}: {g.at[p, c]:,.0f}<br>Participación: {part[c][p]:.1f}%" for p in g.index}
        z, h = por_distrito_desde_provincia({p: float(part[c][p]) for p in g.index}, hov)
        for t in capa_poligonos(z, h, geo=clave, coloraxis="coloraxis"):
            fig.add_trace(t)
        fig.add_trace(capa_bordes(PROV_ID.values(), COLOR_CONTORNO, 1.1, geo=clave))
        top3 = part[c].nlargest(3).index
        fig.add_trace(capa_etiquetas({p: f"<b>{p.title()}</b><br>{part[c][p]:.0f}%" for p in top3},
                                     {p: ("white" if part[c][p] / vmax > 0.5 else "#222222") for p in top3}, CENT_PROV, tam=10, geo=clave))
        ajustar_geo(fig, clave, x=doms[k], y=(0.0, 0.88))
        fig.add_annotation(text=f"<b>{nom}</b>", x=sum(doms[k]) / 2, y=0.96, xref="paper", yref="paper", showarrow=False, font=dict(size=14))
    fig.update_layout(coloraxis=dict(colorscale="Purples", cmin=0, cmax=vmax,
                                     colorbar=dict(title=dict(text="Participación (%)", side="right"), len=0.7, thickness=14, ticksuffix="%")),
                      title=dict(text=f"<b>Misma escala, tres variables</b><br><sup>{subtitulo_mapa(F)}</sup>", x=0.01),
                      height=520, margin=dict(l=10, r=10, t=95, b=20))
    fig.show()
mostrar(vista_mapas_paralelos)

Output()

### 13.8 Gráfico de mancuernas: participación en consumo vs. en facturación

**Indicador:** para cada provincia, un punto verde (participación en consumo) y uno rojo (participación en facturación) unidos por una línea.
**Gestalt:** *continuidad y conexión* (la línea une los dos puntos y el ojo mide su longitud como diferencia), *similitud* (verde y rojo con el mismo significado que en los mapas), *proximidad* (los dos valores de una provincia están en la misma fila) y *figura–fondo* (franja naranja suave tras la provincia elegida).
**Cómo leerlo:** una línea corta significa que la provincia pesa igual en consumo y en facturación; una larga, que no. Si el punto rojo queda a la derecha del verde, factura más de lo que consume.

In [23]:
def vista_mancuernas(**kw):
    F = leer(kw); d, g = tabla_provincias(F); encabezado(F, d)
    g = g[(g["consumo_kwh"] > 0) & (g["facturacion_soles"] > 0)]
    if len(g) < 2:
        print("Pocas provincias con datos para esta combinación de filtros."); return
    pc = 100 * g["consumo_kwh"] / g["consumo_kwh"].sum(); pf = 100 * g["facturacion_soles"] / g["facturacion_soles"].sum()
    orden = list(pc.sort_values().index); y = list(range(len(orden)))
    fig = go.Figure()
    if F["prov"] in orden:
        i = orden.index(F["prov"])
        fig.add_hrect(y0=i - 0.5, y1=i + 0.5, fillcolor=COLOR_SELECCION, opacity=0.15, line_width=0)
    xs, ys = [], []
    for yi, p in zip(y, orden):
        xs += [pc[p], pf[p], None]; ys += [yi, yi, None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#9aa0a6", width=4), hoverinfo="skip", showlegend=False))
    for serie, nom, color in [(pc, "Participación en consumo", COLOR_CONSUMO), (pf, "Participación en facturación", COLOR_FACTURA)]:
        fig.add_trace(go.Scatter(x=[serie[p] for p in orden], y=y, mode="markers", name=nom, marker=dict(color=color, size=13),
                                 hovertemplate="<b>%{text}</b><br>" + nom + ": %{x:.2f}%<extra></extra>", text=[p.title() for p in orden]))
    pad = 0.04 * max(pc.max(), pf.max())
    fig.add_trace(go.Scatter(x=[max(pc[p], pf[p]) + pad for p in orden], y=y, mode="text", textposition="middle right",
                             text=[f"{pf[p] - pc[p]:+.1f} pp" for p in orden], textfont=dict(size=11, color="#555555"),
                             hoverinfo="skip", showlegend=False))
    fig.update_layout(title=dict(text=f"<b>¿Pesa lo mismo en consumo que en facturación?</b><br><sup>{subtitulo_mapa(F)}</sup>", x=0.01),
                      height=max(440, 34 * len(orden) + 150), margin=dict(l=10, r=60, t=95, b=50),
                      xaxis=dict(title="Participación en el total (%)", ticksuffix="%", showgrid=True, gridcolor="#eeeeee", zeroline=False,
                                 range=[0, max(pc.max(), pf.max()) * 1.18]),
                      yaxis=dict(tickmode="array", tickvals=y, ticktext=[p.title() for p in orden], showgrid=False),
                      legend=dict(orientation="h", y=1.02, x=0.0, yanchor="bottom"))
    fig.show()
mostrar(vista_mancuernas)

Output()

### 13.9 Qué mirar al terminar la sección

1. **¿Coinciden los dos mapas (13.3 y 13.4)?** Compare qué provincias quedan en las clases más oscuras; el mapa 13.5 cuantifica las diferencias.
2. **¿Hay provincias grises o rotuladas con aviso?** Revise la salida de 13.1: indica qué distritos no se pueden dibujar y qué porcentaje del consumo y la facturación representan.
3. **¿La conclusión cambia con los filtros?** Todos los mapas responden al mismo panel; pruebe un solo año, una tarifa o una provincia y observe si el patrón se mantiene.